# Análise de Customer Churn em Telecomunicações

## Enquadramento

O customer churn representa o abandono ou cancelamento de um serviço por parte dos clientes.

Este projeto analisa dados de clientes de uma empresa de telecomunicações, com o objetivo de identificar os principais fatores associados ao churn e avaliar o impacto financeiro da perda de clientes.

A análise será realizada em Python e os principais resultados serão posteriormente apresentados num dashboard em Power BI.

## Objetivos

- Calcular a taxa global de churn.
- Identificar os tipos de contrato com maior churn.
- Analisar a relação entre os serviços contratados e o abandono.
- Avaliar o impacto do apoio técnico na retenção.
- Quantificar a receita mensal associada aos clientes perdidos.
- Identificar os segmentos de clientes com maior risco.
- Apresentar recomendações para reduzir o churn.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(r'/content/B2B_SaaS_Churn_Dataset.csv')

print("Dataset carregado com sucesso.")

Dataset carregado com sucesso.


In [3]:
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason,Customer_Tier,Ultimo_Login_Dias
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer,Pro,81
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Electronic check,70.70,151.65,Yes,1,67,2701,Moved,Pro,67
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Electronic check,99.65,820.50,Yes,1,86,5372,Moved,Enterprise,68
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved,Enterprise,88
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices,Enterprise,89


In [4]:
df.shape

(7043, 35)

In [5]:
df.columns

Index(['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
       'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Tenure Months', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value',
       'Churn Score', 'CLTV', 'Churn Reason', 'Customer_Tier',
       'Ultimo_Login_Dias'],
      dtype='object')

# Remoção de colunas antigas

In [6]:
df = df.drop(columns=["Customer_Tier", "Ultimo_Login_Dias"])

In [7]:
df.shape

(7043, 33)

## 2. Auditoria Inicial dos Dados

Antes de iniciar a limpeza e a análise, é necessário avaliar a estrutura e a qualidade do conjunto de dados.

Nesta etapa serão verificados:

* A dimensão do dataset.
* Os tipos de dados de cada coluna.
* A existência de valores em falta.
* A existência de registos duplicados.
* A existência de identificadores de clientes repetidos.
* As principais estatísticas das variáveis numéricas.


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

## Valores em falta

In [9]:
df.isnull().sum()

,0
CustomerID,0
Count,0
Country,0
State,0
City,0
Zip Code,0
Lat Long,0
Latitude,0
Longitude,0
Gender,0


In [10]:
valores_nulos = df.isnull().sum()

valores_nulos[valores_nulos > 0]

,0
Total Charges,11
Churn Reason,5174


##Percentagem de valores em falta

In [11]:
percentagem_nulos = (df.isnull().sum() / len(df)) * 100

percentagem_nulos[percentagem_nulos > 0]

,0
Total Charges,0.156183
Churn Reason,73.463013


##Registos duplicados

In [12]:
duplicados = df.duplicated().sum()

print("Número de registos duplicados:", duplicados)

Número de registos duplicados: 0


## Identificadores de clientes repetidos

In [13]:
df["CustomerID"].duplicated().sum()

np.int64(0)

In [14]:
df.duplicated().sum()

np.int64(0)

## Estatísticas das variáveis numéricas

In [15]:
df.describe()

,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,Churn Value,Churn Score,CLTV
count,7043.0,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7032.000000,7043.000000,7043.000000,7043.000000
mean,1.0,93521.964646,36.282441,-119.798880,32.371149,64.761692,2283.300441,0.265370,58.699418,4400.295755
std,0.0,1865.794555,2.455723,2.157889,24.559481,30.090047,2266.771362,0.441561,21.525131,1183.057152
min,1.0,90001.000000,32.555828,-124.301372,0.000000,18.250000,18.800000,0.000000,5.000000,2003.000000
25%,1.0,92102.000000,34.030915,-121.815412,9.000000,35.500000,401.450000,0.000000,40.000000,3469.000000
50%,1.0,93552.000000,36.391777,-119.730885,29.000000,70.350000,1397.475000,0.000000,61.000000,4527.000000
75%,1.0,95351.000000,38.224869,-118.043237,55.000000,89.850000,3794.737500,1.000000,75.000000,5380.500000
max,1.0,96161.000000,41.962127,-114.192901,72.000000,118.750000,8684.800000,1.000000,100.000000,6500.000000


### Conclusões da Auditoria Inicial

O dataset contém 7.043 registos e, após a remoção das duas variáveis criadas numa versão anterior, apresenta 33 colunas.

A auditoria inicial permitiu concluir que:

* Não existem linhas totalmente duplicadas.
* Não existem identificadores de clientes repetidos.
* A coluna `Total Charges` apresenta 11 valores em falta.
* A coluna `Churn Reason` apresenta 5.174 valores em falta.
* Os valores em falta de `Churn Reason` estão relacionados com clientes que não abandonaram o serviço e serão analisados na etapa de limpeza.
* As principais variáveis numéricas apresentam tipos de dados adequados para a análise.

Na etapa seguinte serão tratados os valores em falta e será validada a coerência das principais variáveis.


## 3. Limpeza e Validação dos Dados

A auditoria inicial identificou valores em falta nas colunas `Total Charges` e `Churn Reason`.

Antes de tratar estes valores, é necessário compreender a razão pela qual estão em falta, de forma a evitar alterações incorretas nos dados.

Nesta etapa serão realizadas as seguintes tarefas:

* Análise dos valores em falta em `Total Charges`.
* Tratamento dos valores em falta em `Churn Reason`.
* Validação dos principais valores numéricos.
* Confirmação da qualidade do dataset após a limpeza.


In [16]:
df[df["Total Charges"].isnull()]

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
2234,4472-LVYGI,1,United States,California,San Bernardino,92408,"34.084909, -117.258107",34.084909,-117.258107,Female,...,Two year,Yes,Bank transfer (automatic),52.55,NaN,No,0,36,2578,NaN
2438,3115-CZMZD,1,United States,California,Independence,93526,"36.869584, -118.189241",36.869584,-118.189241,Male,...,Two year,No,Mailed check,20.25,NaN,No,0,68,5504,NaN
2568,5709-LVOEQ,1,United States,California,San Mateo,94401,"37.590421, -122.306467",37.590421,-122.306467,Female,...,Two year,No,Mailed check,80.85,NaN,No,0,45,2048,NaN
2667,4367-NUYAO,1,United States,California,Cupertino,95014,"37.306612, -122.080621",37.306612,-122.080621,Male,...,Two year,No,Mailed check,25.75,NaN,No,0,48,4950,NaN
2856,1371-DWPAZ,1,United States,California,Redcrest,95569,"40.363446, -123.835041",40.363446,-123.835041,Female,...,Two year,No,Credit card (automatic),56.05,NaN,No,0,30,4740,NaN
4331,7644-OMVMY,1,United States,California,Los Angeles,90029,"34.089953, -118.294824",34.089953,-118.294824,Male,...,Two year,No,Mailed check,19.85,NaN,No,0,53,2019,NaN
4687,3213-VVOLG,1,United States,California,Sun City,92585,"33.739412, -117.173334",33.739412,-117.173334,Male,...,Two year,No,Mailed check,25.35,NaN,No,0,49,2299,NaN
5104,2520-SGTTA,1,United States,California,Ben Lomond,95005,"37.078873, -122.090386",37.078873,-122.090386,Female,...,Two year,No,Mailed check,20.00,NaN,No,0,27,3763,NaN
5719,2923-ARZLG,1,United States,California,La Verne,91750,"34.144703, -117.770299",34.144703,-117.770299,Male,...,One year,Yes,Mailed check,19.70,NaN,No,0,69,4890,NaN
6772,4075-WKNIU,1,United States,California,Bell,90201,"33.970343, -118.171368",33.970343,-118.171368,Female,...,Two year,No,Mailed check,73.35,NaN,No,0,44,2342,NaN


In [17]:
df[df["Total Charges"].isnull()][
    [
        "CustomerID",
        "Tenure Months",
        "Monthly Charges",
        "Total Charges",
        "Contract",
        "Churn Label"
    ]
]

,CustomerID,Tenure Months,Monthly Charges,Total Charges,Contract,Churn Label
2234,4472-LVYGI,0,52.55,NaN,Two year,No
2438,3115-CZMZD,0,20.25,NaN,Two year,No
2568,5709-LVOEQ,0,80.85,NaN,Two year,No
2667,4367-NUYAO,0,25.75,NaN,Two year,No
2856,1371-DWPAZ,0,56.05,NaN,Two year,No
4331,7644-OMVMY,0,19.85,NaN,Two year,No
4687,3213-VVOLG,0,25.35,NaN,Two year,No
5104,2520-SGTTA,0,20.00,NaN,Two year,No
5719,2923-ARZLG,0,19.70,NaN,One year,No
6772,4075-WKNIU,0,73.35,NaN,Two year,No


In [18]:
df["Total Charges"] = df["Total Charges"].fillna(0)

In [19]:
print(
    "Valores em falta em Total Charges:",
    df["Total Charges"].isnull().sum()
)

Valores em falta em Total Charges: 0


### Tratamento de `Total Charges`

Foram identificados 11 clientes com valores em falta na coluna `Total Charges`.

A análise destes registos revelou que todos apresentavam zero meses de permanência (`Tenure Months = 0`). Estes clientes ainda não tinham completado um mês de contrato e, por essa razão, não possuíam um valor acumulado de faturação.

Os valores em falta foram substituídos por zero, preservando todos os registos do dataset.


## Analisar Churn Reason

In [20]:
df[df["Churn Label"] == "No"]["Churn Reason"].isnull().sum()

np.int64(5174)

In [21]:
df[df["Churn Label"] == "Yes"]["Churn Reason"].isnull().sum()

np.int64(0)

In [22]:
df["Churn Reason"] = df["Churn Reason"].fillna(
    "Not Applicable"
)

In [23]:
df["Churn Reason"].isnull().sum()

np.int64(0)

### Tratamento de `Churn Reason`

A coluna `Churn Reason` apresentava 5.174 valores em falta.

Estes valores correspondem aos clientes que não abandonaram o serviço. Como estes clientes permanecem ativos, não existe uma razão de churn associada aos seus registos.

Os valores em falta foram substituídos pela categoria `Not Applicable`, de forma a distinguir uma situação não aplicável de um valor verdadeiramente desconhecido.


In [24]:
df.isnull().sum()

,0
CustomerID,0
Count,0
Country,0
State,0
City,0
Zip Code,0
Lat Long,0
Latitude,0
Longitude,0
Gender,0


In [25]:
valores_nulos = df.isnull().sum()

valores_nulos[valores_nulos > 0]

,0


## Valores negativos

In [26]:
df[
    [
        "Tenure Months",
        "Monthly Charges",
        "Total Charges"
    ]
].min()

,0
Tenure Months,0.00
Monthly Charges,18.25
Total Charges,0.00


## Verificar as categorias principais

In [27]:
df["Churn Label"].value_counts()

,count
Churn Label,
No,5174
Yes,1869


In [28]:
df["Contract"].value_counts()

,count
Contract,
Month-to-month,3875
Two year,1695
One year,1473


In [29]:
df["Tech Support"].value_counts()

,count
Tech Support,
No,3473
Yes,2044
No internet service,1526


In [30]:
df["Payment Method"].value_counts()

,count
Payment Method,
Electronic check,2365
Mailed check,1612
Bank transfer (automatic),1544
Credit card (automatic),1522


### Conclusões da Limpeza e Validação

A limpeza dos dados permitiu corrigir os valores em falta sem eliminar clientes do dataset.

Foram realizadas as seguintes alterações:

* Substituição de 11 valores em falta em `Total Charges` por zero, uma vez que correspondiam a clientes com zero meses de permanência.
* Substituição dos valores em falta em `Churn Reason` pela categoria `Not Applicable`.
* Confirmação da ausência de registos duplicados.
* Confirmação da ausência de identificadores de clientes repetidos.
* Validação dos valores mínimos das principais variáveis financeiras.
* Verificação das categorias utilizadas nas variáveis principais.

Após a limpeza, o dataset mantém os 7.043 clientes e apresenta 33 variáveis, sem valores em falta.


## 4. Análise Global do Churn

Após a limpeza e validação dos dados, inicia-se a análise exploratória do comportamento dos clientes.

A primeira etapa consiste em calcular:

* O número total de clientes.
* O número de clientes ativos.
* O número de clientes que abandonaram o serviço.
* A taxa global de churn.

Estes indicadores permitem compreender a dimensão geral do problema antes de analisar os segmentos e fatores associados ao abandono.


In [31]:
total_clientes = len(df)

In [32]:
clientes_ativos = (
    df["Churn Label"] == "No"
).sum()

In [33]:
clientes_perdidos = (
    df["Churn Label"] == "Yes"
).sum()

In [34]:
taxa_churn = (
    clientes_perdidos / total_clientes
) * 100

In [35]:
print("Total de clientes:", total_clientes)
print("Clientes ativos:", clientes_ativos)
print("Clientes perdidos:", clientes_perdidos)
print("Taxa global de churn:", round(taxa_churn, 2), "%")

Total de clientes: 7043
Clientes ativos: 5174
Clientes perdidos: 1869
Taxa global de churn: 26.54 %


### Interpretação

A empresa apresenta um total de 7.043 clientes.

Destes:

* 5.174 permanecem ativos.
* 1.869 abandonaram o serviço.
* A taxa global de churn é de 26,54%.

Isto significa que aproximadamente um em cada quatro clientes presentes no dataset abandonou a empresa.

Este resultado revela um nível de churn relevante e justifica uma análise mais detalhada dos contratos, serviços e características dos clientes associados ao abandono.


In [36]:
df["Churn Label"].value_counts()

,count
Churn Label,
No,5174
Yes,1869


## 5. Impacto Financeiro Global do Churn

Para além da perda de clientes, o churn representa também uma perda de receita para a empresa.

Nesta secção será calculada a faturação mensal associada aos clientes que abandonaram o serviço e o respetivo impacto anualizado.

O valor anualizado representa uma estimativa obtida através da multiplicação da faturação mensal por 12 meses.


In [37]:
clientes_churn = df[df["Churn Label"] == "Yes"]

In [38]:
clientes_churn.shape

(1869, 33)

##Calcular a faturação mensal perdida

In [39]:
receita_mensal_perdida = clientes_churn["Monthly Charges"].sum()

print(
    "Receita mensal associada aos clientes perdidos:",
    round(receita_mensal_perdida, 2),
    "€"
)

Receita mensal associada aos clientes perdidos: 139130.85 €


## Anual

In [40]:
impacto_anualizado = receita_mensal_perdida * 12

print(
    "Impacto anualizado estimado:",
    round(impacto_anualizado, 2),
    "€"
)

Impacto anualizado estimado: 1669570.2 €


## Mensalidade média dos clientes perdidos

In [41]:
mensalidade_media_churn = clientes_churn["Monthly Charges"].mean()

print(
    "Mensalidade média dos clientes perdidos:",
    round(mensalidade_media_churn, 2),
    "€"
)

Mensalidade média dos clientes perdidos: 74.44 €


In [42]:
clientes_ativos_df = df[df["Churn Label"] == "No"]

mensalidade_media_ativos = clientes_ativos_df[
    "Monthly Charges"
].mean()

print(
    "Mensalidade média dos clientes ativos:",
    round(mensalidade_media_ativos, 2),
    "€"
)

Mensalidade média dos clientes ativos: 61.27 €


## Comparação das mensalidades médias

In [43]:
print(
    "Mensalidade média dos clientes ativos:",
    round(mensalidade_media_ativos, 2),
    "€"
)

print(
    "Mensalidade média dos clientes perdidos:",
    round(mensalidade_media_churn, 2),
    "€"
)

Mensalidade média dos clientes ativos: 61.27 €
Mensalidade média dos clientes perdidos: 74.44 €


### Interpretação do Impacto Financeiro

Os 1.869 clientes que abandonaram o serviço representavam 139.130,85 € em faturação mensal.

Caso este valor se mantivesse constante durante 12 meses, corresponderia a um impacto anualizado estimado de aproximadamente 1,67 milhões de euros.

A mensalidade média dos clientes perdidos é também superior à mensalidade média dos clientes ativos. Este resultado sugere que o churn apresenta não apenas um impacto no número de clientes, mas também um impacto financeiro relevante.

Contudo, esta análise representa uma estimativa baseada nas mensalidades registadas no dataset. O conjunto de dados não inclui datas de cancelamento nem um histórico mensal de receita que permita calcular a perda financeira efetiva ao longo do tempo.


## 6. Churn por Tipo de Contrato

O tipo de contrato pode influenciar a permanência dos clientes na empresa.

Nesta secção será analisada a taxa de churn dos clientes com contratos mensais, anuais e bienais, com o objetivo de identificar os contratos associados a um maior risco de abandono.


In [44]:
df["Contract"].value_counts()

,count
Contract,
Month-to-month,3875
Two year,1695
One year,1473


## coluna numérica para o churn

In [45]:
df["Churn_Numero"] = df["Churn Label"].map({
    "No": 0,
    "Yes": 1
})

In [46]:
df[
    [
        "Churn Label",
        "Churn_Numero"
    ]
].head()

,Churn Label,Churn_Numero
0,Yes,1
1,Yes,1
2,Yes,1
3,Yes,1
4,Yes,1


In [47]:
df["Churn_Numero"].value_counts()

,count
Churn_Numero,
0,5174
1,1869


## taxa de churn por contrato

In [48]:
churn_contrato = df.groupby(
    "Contract"
)["Churn_Numero"].mean() * 100

In [49]:
churn_contrato

,Churn_Numero
Contract,
Month-to-month,42.709677
One year,11.269518
Two year,2.831858


In [50]:
churn_contrato = churn_contrato.round(2)

churn_contrato

,Churn_Numero
Contract,
Month-to-month,42.71
One year,11.27
Two year,2.83


In [51]:
total_por_contrato = df.groupby(
    "Contract"
)["CustomerID"].count()

In [52]:
churn_por_contrato = df.groupby(
    "Contract"
)["Churn_Numero"].sum()

In [53]:
tabela_contrato = pd.DataFrame({
    "Total de Clientes": total_por_contrato,
    "Clientes Perdidos": churn_por_contrato,
    "Taxa de Churn (%)": churn_contrato
})

tabela_contrato

,Total de Clientes,Clientes Perdidos,Taxa de Churn (%)
Contract,,,
Month-to-month,3875,1655,42.71
One year,1473,166,11.27
Two year,1695,48,2.83


## Receita mensal perdida por contrato

In [54]:
receita_perdida_contrato = clientes_churn.groupby(
    "Contract"
)["Monthly Charges"].sum()

In [55]:
receita_perdida_contrato = (
    receita_perdida_contrato.round(2)
)

receita_perdida_contrato

,Monthly Charges
Contract,
Month-to-month,120847.10
One year,14118.45
Two year,4165.30


In [56]:
tabela_contrato[
    "Receita Mensal Perdida (€)"
] = receita_perdida_contrato

tabela_contrato

,Total de Clientes,Clientes Perdidos,Taxa de Churn (%),Receita Mensal Perdida (€)
Contract,,,,
Month-to-month,3875,1655,42.71,120847.10
One year,1473,166,11.27,14118.45
Two year,1695,48,2.83,4165.30


## Percentagem da receita perdida associada aos contratos mensais

In [57]:
receita_month_to_month = receita_perdida_contrato[
    "Month-to-month"
]

percentagem_receita_mensal = (
    receita_month_to_month /
    receita_mensal_perdida
) * 100

print(
    "Percentagem da receita mensal perdida associada a contratos mensais:",
    round(percentagem_receita_mensal, 2),
    "%"
)

Percentagem da receita mensal perdida associada a contratos mensais: 86.86 %


### Interpretação do Churn por Tipo de Contrato

Os clientes com contratos `Month-to-month` apresentam uma taxa de churn de 42,71%, consideravelmente superior à taxa observada nos contratos de um ano e dois anos.

Em comparação:

* Os contratos de um ano apresentam uma taxa de churn de 11,27%.
* Os contratos de dois anos apresentam uma taxa de churn de apenas 2,83%.

Os contratos mensais também concentram 1.655 dos 1.869 clientes perdidos e aproximadamente 87% da receita mensal associada ao churn.

Estes resultados demonstram uma forte associação entre contratos de curta duração e maior risco de abandono. Os clientes com contratos mensais devem, por isso, representar um dos principais públicos das estratégias de retenção.

Contudo, esta relação não prova que o tipo de contrato seja, isoladamente, a causa do churn. Outros fatores, como o tempo de permanência, os serviços contratados e o apoio técnico, também podem influenciar o comportamento dos clientes.


## 7. Churn por Apoio Técnico

O apoio técnico pode desempenhar um papel importante na experiência e retenção dos clientes.

Nesta secção será analisada a taxa de churn dos clientes com e sem o serviço de `Tech Support`.

Também será realizada uma análise específica aos clientes com contratos mensais, uma vez que este grupo apresentou o maior risco de abandono na secção anterior.


In [58]:
df["Tech Support"].value_counts()

,count
Tech Support,
No,3473
Yes,2044
No internet service,1526


## taxa de churn por apoio técnico

In [59]:
taxa_churn_tech = df.groupby(
    "Tech Support"
)["Churn_Numero"].mean() * 100

taxa_churn_tech = taxa_churn_tech.round(2)

taxa_churn_tech

,Churn_Numero
Tech Support,
No,41.64
No internet service,7.40
Yes,15.17


In [60]:
total_por_tech = df.groupby(
    "Tech Support"
)["CustomerID"].count()

perdidos_por_tech = df.groupby(
    "Tech Support"
)["Churn_Numero"].sum()

In [61]:
tabela_tech = pd.DataFrame({
    "Total de Clientes": total_por_tech,
    "Clientes Perdidos": perdidos_por_tech,
    "Taxa de Churn (%)": taxa_churn_tech
})

tabela_tech

,Total de Clientes,Clientes Perdidos,Taxa de Churn (%)
Tech Support,,,
No,3473,1446,41.64
No internet service,1526,113,7.40
Yes,2044,310,15.17


In [62]:
clientes_mensais = df[
    df["Contract"] == "Month-to-month"
]

In [63]:
clientes_mensais.shape

(3875, 34)

## Churn por apoio técnico nos contratos mensais

In [64]:
taxa_tech_mensais = clientes_mensais.groupby(
    "Tech Support"
)["Churn_Numero"].mean() * 100

taxa_tech_mensais = taxa_tech_mensais.round(2)

taxa_tech_mensais

,Churn_Numero
Tech Support,
No,50.37
No internet service,18.89
Yes,30.70


In [65]:
total_tech_mensais = clientes_mensais.groupby(
    "Tech Support"
)["CustomerID"].count()

perdidos_tech_mensais = clientes_mensais.groupby(
    "Tech Support"
)["Churn_Numero"].sum()

In [66]:
tabela_tech_mensais = pd.DataFrame({
    "Total de Clientes": total_tech_mensais,
    "Clientes Perdidos": perdidos_tech_mensais,
    "Taxa de Churn (%)": taxa_tech_mensais
})

tabela_tech_mensais

,Total de Clientes,Clientes Perdidos,Taxa de Churn (%)
Tech Support,,,
No,2680,1350,50.37
No internet service,524,99,18.89
Yes,671,206,30.70


## diferença entre os dois grupos

In [67]:
diferenca_tech = (
    taxa_tech_mensais["No"] -
    taxa_tech_mensais["Yes"]
)

print(
    "Diferença na taxa de churn:",
    round(diferenca_tech, 2),
    "pontos percentuais"
)

Diferença na taxa de churn: 19.67 pontos percentuais


### Interpretação do Churn por Apoio Técnico

A análise global mostra que os clientes sem `Tech Support` apresentam uma taxa de churn de 41,64%, enquanto os clientes com este serviço apresentam uma taxa de 15,17%.

Contudo, estes grupos incluem clientes com diferentes tipos de contrato. Para realizar uma comparação mais adequada, foi analisado separadamente o grupo com contratos `Month-to-month`.

Entre estes clientes:

* A taxa de churn é de 50,37% quando não existe `Tech Support`.
* A taxa de churn é de 30,70% quando o serviço está ativo.
* A diferença entre os dois grupos é de 19,67 pontos percentuais.

Os resultados demonstram uma associação entre a presença de apoio técnico e uma menor taxa de churn, mesmo dentro do segmento de contratos mensais.

Esta relação não prova que o apoio técnico seja, isoladamente, a causa da redução do churn. Outros fatores relacionados com o perfil dos clientes e os serviços contratados também podem influenciar os resultados.


## 8. Churn por Tempo de Permanência

O tempo de permanência, representado pela variável `Tenure Months`, indica há quantos meses cada cliente mantém uma relação com a empresa.

Nesta secção será comparado o tempo de permanência dos clientes ativos com o dos clientes que abandonaram o serviço.

O objetivo é perceber se o churn ocorre com maior frequência entre clientes recentes ou entre clientes com uma relação mais longa com a empresa.


In [68]:
df["Tenure Months"].describe()

,Tenure Months
count,7043.000000
mean,32.371149
std,24.559481
min,0.000000
25%,9.000000
50%,29.000000
75%,55.000000
max,72.000000


In [69]:
tenure_medio = df.groupby(
    "Churn Label"
)["Tenure Months"].mean()

tenure_medio.round(2)

,Tenure Months
Churn Label,
No,37.57
Yes,17.98


In [70]:
tenure_mediano = df.groupby(
    "Churn Label"
)["Tenure Months"].median()

tenure_mediano

,Tenure Months
Churn Label,
No,38.0
Yes,10.0


In [71]:
tabela_tenure = pd.DataFrame({
    "Tenure Médio": tenure_medio.round(2),
    "Tenure Mediano": tenure_mediano
})

tabela_tenure

,Tenure Médio,Tenure Mediano
Churn Label,,
No,37.57,38.0
Yes,17.98,10.0


## Criar grupos de permanência

In [72]:
df["Grupo_Tenure"] = pd.cut(
    df["Tenure Months"],
    bins=[-1, 12, 24, 48, 72],
    labels=[
        "0-12 meses",
        "13-24 meses",
        "25-48 meses",
        "49-72 meses"
    ]
)

In [73]:
df[
    [
        "Tenure Months",
        "Grupo_Tenure"
    ]
].head(10)

,Tenure Months,Grupo_Tenure
0,2,0-12 meses
1,2,0-12 meses
2,8,0-12 meses
3,28,25-48 meses
4,49,49-72 meses
5,10,0-12 meses
6,1,0-12 meses
7,1,0-12 meses
8,47,25-48 meses
9,1,0-12 meses


In [74]:
df["Grupo_Tenure"].value_counts()

,count
Grupo_Tenure,
49-72 meses,2239
0-12 meses,2186
25-48 meses,1594
13-24 meses,1024


## Taxa de churn por grupo de permanência

In [75]:
taxa_churn_tenure = df.groupby(
    "Grupo_Tenure",
    observed=False
)["Churn_Numero"].mean() * 100

taxa_churn_tenure = taxa_churn_tenure.round(2)

taxa_churn_tenure

,Churn_Numero
Grupo_Tenure,
0-12 meses,47.44
13-24 meses,28.71
25-48 meses,20.39
49-72 meses,9.51


In [76]:
total_por_tenure = df.groupby(
    "Grupo_Tenure",
    observed=False
)["CustomerID"].count()

In [77]:
perdidos_por_tenure = df.groupby(
    "Grupo_Tenure",
    observed=False
)["Churn_Numero"].sum()

In [78]:
tabela_tenure_grupos = pd.DataFrame({
    "Total de Clientes": total_por_tenure,
    "Clientes Perdidos": perdidos_por_tenure,
    "Taxa de Churn (%)": taxa_churn_tenure
})

tabela_tenure_grupos

,Total de Clientes,Clientes Perdidos,Taxa de Churn (%)
Grupo_Tenure,,,
0-12 meses,2186,1037,47.44
13-24 meses,1024,294,28.71
25-48 meses,1594,325,20.39
49-72 meses,2239,213,9.51


## Clientes perdidos no primeiro ano

In [79]:
churn_primeiro_ano = clientes_churn[
    clientes_churn["Tenure Months"] <= 12
].shape[0]

print(
    "Clientes perdidos com até 12 meses:",
    churn_primeiro_ano
)

Clientes perdidos com até 12 meses: 1037


In [80]:
percentagem_churn_primeiro_ano = (
    churn_primeiro_ano /
    clientes_perdidos
) * 100

print(
    "Percentagem dos clientes perdidos com até 12 meses:",
    round(percentagem_churn_primeiro_ano, 2),
    "%"
)

Percentagem dos clientes perdidos com até 12 meses: 55.48 %


### Interpretação do Tempo de Permanência

Os clientes que abandonaram apresentam um tempo médio e mediano de permanência inferior ao dos clientes que continuam ativos.

A análise por grupos confirma que o churn é mais elevado durante as fases iniciais da relação com a empresa, sobretudo entre os clientes com até 12 meses de permanência.

À medida que o tempo de permanência aumenta, a taxa de churn tende a diminuir. Este resultado sugere que os primeiros meses do contrato representam um período particularmente importante para a retenção.

A empresa deve, por isso, reforçar o acompanhamento dos novos clientes, avaliar a qualidade do processo inicial de adesão e identificar sinais de insatisfação antes do cancelamento.

Apesar desta associação, o tempo de permanência não deve ser analisado isoladamente. Os clientes recentes também podem apresentar contratos mensais, menor adesão a serviços de apoio e diferentes níveis de faturação.


## 9. Churn por Método de Pagamento

O método de pagamento pode estar associado à experiência e ao comportamento dos clientes.

Nesta secção será calculada a taxa de churn de cada método de pagamento, com o objetivo de identificar os grupos que apresentam maior risco de abandono.


In [81]:
df["Payment Method"].value_counts()

,count
Payment Method,
Electronic check,2365
Mailed check,1612
Bank transfer (automatic),1544
Credit card (automatic),1522


In [82]:
taxa_churn_pagamento = df.groupby(
    "Payment Method"
)["Churn_Numero"].mean() * 100

taxa_churn_pagamento = taxa_churn_pagamento.round(2)

taxa_churn_pagamento

,Churn_Numero
Payment Method,
Bank transfer (automatic),16.71
Credit card (automatic),15.24
Electronic check,45.29
Mailed check,19.11


In [83]:
total_por_pagamento = df.groupby(
    "Payment Method"
)["CustomerID"].count()

In [84]:
perdidos_por_pagamento = df.groupby(
    "Payment Method"
)["Churn_Numero"].sum()

In [85]:
tabela_pagamento = pd.DataFrame({
    "Total de Clientes": total_por_pagamento,
    "Clientes Perdidos": perdidos_por_pagamento,
    "Taxa de Churn (%)": taxa_churn_pagamento
})

tabela_pagamento

,Total de Clientes,Clientes Perdidos,Taxa de Churn (%)
Payment Method,,,
Bank transfer (automatic),1544,258,16.71
Credit card (automatic),1522,232,15.24
Electronic check,2365,1071,45.29
Mailed check,1612,308,19.11


### Interpretação do Churn por Método de Pagamento

O método `Electronic check` apresenta a maior taxa de churn, com 45,29%. Neste grupo, 1.071 dos 2.365 clientes abandonaram o serviço.

Os restantes métodos apresentam taxas bastante inferiores:

* `Mailed check`: 19,11%.
* `Bank transfer (automatic)`: 16,71%.
* `Credit card (automatic)`: 15,24%.

Os dois métodos de pagamento automático apresentam as taxas de churn mais baixas. Em contrapartida, a taxa associada ao `Electronic check` é quase três vezes superior à observada no pagamento automático por cartão de crédito.

Estes resultados identificam o `Electronic check` como um indicador relevante de risco. Contudo, o método de pagamento não deve ser considerado uma causa isolada do churn, uma vez que pode estar associado a outros fatores, como o tipo de contrato e o tempo de permanência.



## 10. Segmentação dos Clientes por Valor Mensal

Para avaliar o impacto financeiro do churn, os clientes serão agrupados de acordo com o valor da sua mensalidade.

Foram definidos três segmentos:

* **Baixo Valor:** mensalidade inferior a 40 €.
* **Valor Médio:** mensalidade entre 40 € e 80 €.
* **Alto Valor:** mensalidade superior a 80 €.

Esta segmentação não representa planos comerciais da empresa. Trata-se de uma classificação criada exclusivamente para esta análise, com o objetivo de comparar o risco de churn e a receita associada a diferentes níveis de faturação mensal.


In [86]:
def classificar_valor(mensalidade):

    if mensalidade < 40:
        return "Baixo Valor"

    elif mensalidade <= 80:
        return "Valor Médio"

    else:
        return "Alto Valor"

In [87]:
df["Segmento_Valor"] = df[
    "Monthly Charges"
].apply(classificar_valor)

In [89]:
df[
    [
        "Monthly Charges",
        "Segmento_Valor"
    ]
].head(10)

,Monthly Charges,Segmento_Valor
0,53.85,Valor Médio
1,70.70,Valor Médio
2,99.65,Alto Valor
3,104.80,Alto Valor
4,103.70,Alto Valor
5,55.20,Valor Médio
6,39.65,Baixo Valor
7,20.15,Baixo Valor
8,99.35,Alto Valor
9,30.20,Baixo Valor


In [90]:
df["Segmento_Valor"].value_counts()

,count
Segmento_Valor,
Alto Valor,2666
Valor Médio,2540
Baixo Valor,1837


## Taxa de churn por segmento

In [91]:
taxa_churn_segmento = df.groupby(
    "Segmento_Valor"
)["Churn_Numero"].mean() * 100

taxa_churn_segmento = taxa_churn_segmento.round(2)

taxa_churn_segmento

,Churn_Numero
Segmento_Valor,
Alto Valor,33.98
Baixo Valor,11.59
Valor Médio,29.53


In [92]:
total_por_segmento = df.groupby(
    "Segmento_Valor"
)["CustomerID"].count()

In [93]:
perdidos_por_segmento = df.groupby(
    "Segmento_Valor"
)["Churn_Numero"].sum()

In [95]:
clientes_churn = df[
    df["Churn Label"] == "Yes"
]

In [96]:
receita_perdida_segmento = clientes_churn.groupby(
    "Segmento_Valor"
)["Monthly Charges"].sum()

receita_perdida_segmento = (
    receita_perdida_segmento.round(2)
)

In [97]:
tabela_segmentos = pd.DataFrame({
    "Total de Clientes": total_por_segmento,
    "Clientes Perdidos": perdidos_por_segmento,
    "Taxa de Churn (%)": taxa_churn_segmento,
    "Receita Mensal Perdida (€)": receita_perdida_segmento
})

tabela_segmentos

,Total de Clientes,Clientes Perdidos,Taxa de Churn (%),Receita Mensal Perdida (€)
Segmento_Valor,,,,
Alto Valor,2666,906,33.98,85349.35
Baixo Valor,1837,213,11.59,5343.20
Valor Médio,2540,750,29.53,48438.30


In [98]:
receita_alto_valor = receita_perdida_segmento[
    "Alto Valor"
]

percentagem_alto_valor = (
    receita_alto_valor /
    receita_mensal_perdida
) * 100

print(
    "Percentagem da receita mensal perdida associada ao segmento Alto Valor:",
    round(percentagem_alto_valor, 2),
    "%"
)

Percentagem da receita mensal perdida associada ao segmento Alto Valor: 61.34 %


### Interpretação da Segmentação por Valor

O segmento `Alto Valor`, composto por clientes com mensalidades superiores a 80 €, apresenta a maior taxa de churn, com 33,98%.

Este segmento concentra 906 clientes perdidos e 85.349,35 € de receita mensal associada ao churn, o que representa aproximadamente 61% da receita mensal perdida.

O segmento `Valor Médio` apresenta também uma taxa de churn elevada, de 29,53%, enquanto o segmento `Baixo Valor` regista uma taxa consideravelmente inferior, de 11,59%.

Os resultados mostram que os clientes com mensalidades mais elevadas combinam maior risco de abandono com maior impacto financeiro. Por essa razão, devem representar uma prioridade nas estratégias de retenção.

Contudo, uma mensalidade elevada não deve ser interpretada isoladamente como causa do churn. Estes clientes podem também possuir mais serviços, contratos mensais ou outros fatores associados ao abandono.


## 11. Principais Conclusões

A análise de 7.043 clientes permitiu identificar vários padrões associados ao abandono do serviço.

### Taxa Global de Churn

A empresa apresenta uma taxa global de churn de **26,54%**.

Dos 7.043 clientes analisados:

* 5.174 permanecem ativos.
* 1.869 abandonaram o serviço.

Isto significa que aproximadamente um em cada quatro clientes presentes no dataset abandonou a empresa.

### Impacto Financeiro

Os clientes que abandonaram representavam **139.130,85 € em faturação mensal**.

Este valor corresponde a um impacto anualizado estimado de aproximadamente **1,67 milhões de euros**, caso o nível de faturação mensal se mantivesse constante durante 12 meses.

### Tipo de Contrato

Os contratos `Month-to-month` apresentam a maior taxa de churn, com **42,71%**.

Em comparação:

* Contratos de um ano: 11,27%.
* Contratos de dois anos: 2,83%.

Os contratos mensais concentram também aproximadamente **87% da receita mensal associada ao churn**.

### Apoio Técnico

Entre os clientes com contratos mensais:

* Sem `Tech Support`, a taxa de churn é de 50,37%.
* Com `Tech Support`, a taxa de churn é de 30,70%.

A diferença de 19,67 pontos percentuais demonstra uma associação relevante entre a presença de apoio técnico e uma menor taxa de abandono.

### Tempo de Permanência

Os clientes que abandonaram apresentam um tempo de permanência médio e mediano inferior ao dos clientes ativos.

O churn é particularmente elevado durante os primeiros meses da relação com a empresa, o que demonstra a importância do acompanhamento inicial dos novos clientes.

### Método de Pagamento

O método `Electronic check` apresenta uma taxa de churn de **45,29%**, bastante superior às restantes formas de pagamento.

Os métodos automáticos apresentam taxas mais reduzidas:

* Transferência bancária automática: 16,71%.
* Cartão de crédito automático: 15,24%.

### Segmentação por Valor

O segmento `Alto Valor`, composto por clientes com mensalidades superiores a 80 €, apresenta uma taxa de churn de **33,98%**.

Este segmento concentra:

* 906 clientes perdidos.
* 85.349,35 € de receita mensal associada ao churn.
* Aproximadamente 61% da receita mensal perdida.

Este grupo combina um elevado risco de abandono com um elevado impacto financeiro, devendo representar uma prioridade nas estratégias de retenção.


## 12. Recomendações Estratégicas

Com base nos resultados da análise, são propostas as seguintes ações:

### 1. Priorizar Clientes com Contratos Mensais

Os clientes com contratos `Month-to-month` apresentam a maior taxa de churn e concentram a maior parte da receita mensal perdida.

A empresa deve desenvolver campanhas que incentivem a passagem para contratos anuais ou bienais, através de benefícios como:

* Descontos temporários.
* Preços fixos durante o período do contrato.
* Serviços adicionais incluídos.
* Benefícios de fidelização.

### 2. Reforçar o Apoio Técnico

A presença de `Tech Support` está associada a taxas de churn inferiores, especialmente entre os clientes com contratos mensais.

A empresa deve promover este serviço durante os primeiros meses do contrato e facilitar o contacto com o apoio técnico sempre que sejam identificados problemas de utilização ou insatisfação.

### 3. Melhorar o Acompanhamento Inicial

Os clientes com menor tempo de permanência apresentam maior risco de abandono.

Por esse motivo, a empresa deve reforçar o acompanhamento durante o primeiro ano, através de:

* Contactos de acompanhamento após a adesão.
* Questionários de satisfação.
* Comunicações educativas sobre os serviços contratados.
* Identificação rápida de dificuldades técnicas.
* Ofertas personalizadas para clientes em risco.

### 4. Incentivar Métodos de Pagamento Automáticos

O método `Electronic check` apresenta uma taxa de churn muito superior aos métodos automáticos.

A empresa pode incentivar a adesão à transferência bancária automática ou ao cartão de crédito automático através de processos de alteração simples e benefícios específicos.

Contudo, antes de implementar esta medida, deve confirmar se o método de pagamento está associado a outros fatores, como o tipo de contrato ou o perfil do cliente.

### 5. Proteger os Clientes de Alto Valor

Os clientes com mensalidades superiores a 80 € representam a maior parcela da receita mensal perdida.

A empresa deve criar ações específicas para este grupo, como:

* Atendimento prioritário.
* Revisão periódica do plano contratado.
* Ofertas personalizadas.
* Benefícios de fidelização.
* Contacto preventivo perante sinais de insatisfação.

### 6. Desenvolver um Sistema de Identificação de Risco

Os resultados desta análise podem servir de base para o desenvolvimento futuro de um modelo preditivo de churn.

Este modelo poderia combinar variáveis como:

* Tipo de contrato.
* Tempo de permanência.
* Apoio técnico.
* Método de pagamento.
* Valor da mensalidade.
* Serviços contratados.

O objetivo seria identificar clientes com maior probabilidade de abandono antes do cancelamento e permitir uma intervenção antecipada.


## 13. Limitações da Análise

Apesar de permitir identificar padrões relevantes, esta análise apresenta algumas limitações:

* O dataset representa uma fotografia dos clientes num determinado momento e não contém uma evolução histórica mensal.
* Não existem datas específicas de cancelamento.
* O impacto anualizado foi estimado através da multiplicação da faturação mensal por 12.
* A análise identifica associações entre variáveis, mas não permite provar relações de causa e efeito.
* A segmentação por valor foi criada especificamente para este projeto e não representa planos comerciais reais da empresa.
* Não foram considerados custos de aquisição, margens, descontos ou custos operacionais.
* O dataset não contém informação detalhada sobre interações com o serviço de apoio, reclamações ou níveis de satisfação.
* Não foi desenvolvido um modelo preditivo de churn nesta fase do projeto.

Estas limitações devem ser consideradas na interpretação dos resultados e na aplicação das recomendações.


In [99]:
print("Número de clientes:", df.shape[0])
print("Número de colunas:", df.shape[1])

Número de clientes: 7043
Número de colunas: 36


In [100]:
print(
    "Total de valores em falta:",
    df.isnull().sum().sum()
)

Total de valores em falta: 0


In [101]:
df[
    [
        "Churn Label",
        "Churn_Numero",
        "Tenure Months",
        "Grupo_Tenure",
        "Monthly Charges",
        "Segmento_Valor"
    ]
].head()

,Churn Label,Churn_Numero,Tenure Months,Grupo_Tenure,Monthly Charges,Segmento_Valor
0,Yes,1,2,0-12 meses,53.85,Valor Médio
1,Yes,1,2,0-12 meses,70.70,Valor Médio
2,Yes,1,8,0-12 meses,99.65,Alto Valor
3,Yes,1,28,25-48 meses,104.80,Alto Valor
4,Yes,1,49,49-72 meses,103.70,Alto Valor


In [102]:
df.to_csv(
    "/content/telco_churn_limpo.csv",
    index=False
)

print("Dataset exportado com sucesso.")

Dataset exportado com sucesso.


## Conclusão Final

Este projeto permitiu transformar dados de clientes numa análise orientada para a retenção e proteção da receita.

Os resultados mostram que o churn está particularmente associado a contratos mensais, ausência de apoio técnico, menor tempo de permanência, utilização de cheque eletrónico e mensalidades mais elevadas.

A análise permitiu identificar grupos prioritários para ações de retenção e preparar uma base de dados limpa e enriquecida para utilização num dashboard interativo em Power BI.